# AF2-SFS-CUE direct/fresh — single-arm seed-42 screen

Hanya melatih **AF2SFSCUE1** satu kali dari official `yolo26n.pt`. Model memakai SFS dan satu decoder factorized CUE+SPDS: `g_hat` diawasi oleh `g` serta `x*g` tanpa decoder kedua. Tidak memakai checkpoint kopi, tidak menjalankan empat-arm ablation, dan tidak membuka test. Historical AF2DIRECT hanya dipakai sebagai screening; matched-runtime control baru dijalankan jika kandidat ini menang besar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import importlib, json, os, shutil, subprocess, sys, tarfile, time, zipfile
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='codex/af2-sfs-cue-direct'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result=subprocess.run(clone,cwd='/content')
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    time.sleep(2)
else: raise RuntimeError('git clone gagal tiga kali')
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True,cwd='/content')
SRC=str(REPO/'src')
if SRC not in sys.path: sys.path.insert(0,SRC)
importlib.invalidate_caches()
for key in list(sys.modules):
    if key=='coffee_detector' or key.startswith('coffee_detector.'): sys.modules.pop(key,None)
os.chdir(REPO)
import coffee_detector, torch
assert Path(coffee_detector.__file__).resolve().is_relative_to(REPO.resolve()),coffee_detector.__file__
assert torch.cuda.is_available(),'Aktifkan GPU Colab.'
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
PROJECT=resolve_drive_project_root(required_relative_paths=(
 'bundles/faruq-development-v3-grouped.tar',
 'bundles/af2-direct-from-pretrained-seed42-state.zip',
))
ARCHIVE=require_project_artifact(PROJECT,'bundles/faruq-development-v3-grouped.tar')
AF2_STATE=require_project_artifact(PROJECT,'bundles/af2-direct-from-pretrained-seed42-state.zip')
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as stream: stream.extractall('/content',filter='data')
STATE=Path('/content/af2-direct-state')
if STATE.exists(): shutil.rmtree(STATE)
with zipfile.ZipFile(AF2_STATE) as stream: stream.extractall(STATE)
summaries=sorted(STATE.rglob('af2_direct_seed42_summary.json'))
if len(summaries)!=1: raise FileNotFoundError(f'Harus satu historical summary; ditemukan {summaries}')
HISTORICAL=summaries[0]
GROUPED=DATA/'faruq_grouped_summary.json'
assert GROUPED.is_file() and not (DATA/'test').exists()
from ultralytics import YOLO
_=YOLO('yolo26n.pt')
PRETRAINED=(REPO/'yolo26n.pt').resolve()
assert PRETRAINED.is_file()
OUTPUT=PROJECT/'experiments/faruq-v3-af2-sfs-cue-direct-v1'
OUTPUT.mkdir(parents=True,exist_ok=True)
print('DATA:',DATA)
print('HISTORICAL:',HISTORICAL)
print('OUTPUT:',OUTPUT)


In [ ]:
from coffee_detector.af2_sfs_cue.audit import run_af2_sfs_cue_direct_static_audit
STATIC=OUTPUT/'static_audit.json'
cached=json.loads(STATIC.read_text()) if STATIC.is_file() else {}
if cached.get('decision')=='PASS' and 'initial_detector_output_numerically_equivalent' in cached.get('gates',{}):
    audit=cached
else:
    audit=run_af2_sfs_cue_direct_static_audit(PRETRAINED,STATIC,seed=42,device='0')
print('PARAMETERS:',{'training_only_added':audit['training_only_added_parameters'],'inference_added':audit['inference_added_parameters']})
print('INITIAL DIFF:',audit['initial_output_max_abs_diff'])
print('GATES:',audit['gates'])
print('DECISION:',audit['decision'])
assert audit['decision']=='PASS','STOP: static audit gagal; jangan training.'


In [ ]:
ARM='AF2SFSCUE1'; LOG=OUTPUT/f'{ARM}_seed42_run.log'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_sfs_cue_direct',
 '--data-root',str(DATA),'--grouped-summary',str(GROUPED),
 '--pretrained-checkpoint',str(PRETRAINED),'--historical-af2direct-summary',str(HISTORICAL),
 '--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
print('START/RESUME:',ARM,'| log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream:
    process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT,text=True)
last_status=None
while process.poll() is None:
    csv_path=OUTPUT/ARM/f'{ARM}_seed42/results.csv'
    epochs=max(0,sum(1 for _ in csv_path.open())-1) if csv_path.is_file() else 0
    status=f'{ARM}: {epochs}/50 epoch tercatat'
    if status!=last_status: print(status,flush=True); last_status=status
    time.sleep(60)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-150:]))
    raise RuntimeError(f'{ARM} gagal: {process.returncode}')
SUMMARY=OUTPUT/'af2_sfs_cue_direct_seed42_summary.json'
summary=json.loads(SUMMARY.read_text())
print(json.dumps({'values':{'AF2DIRECT':summary['historical_af2direct']['metrics'],ARM:summary['candidate']['metrics']},'screen':summary['screen'],'test':summary['test_images_accessed']},indent=2))
print('Kirim values dan screen. Jangan membuka test.')
